In [1]:
#setup
%load_ext autoreload
%autoreload 2

from pathlib import Path

import ccxt
import pandas as pd

from cryptosense.data import (
    candles_to_dataframe,
    drop_incomplete_candles,
    fetch_ohlcv_history,
    load_processed,
    save_dataset,
    validate_ohlcv,
)

In [2]:
#configuration
WATCHLIST = ["BTC/USDT", "ETH/USDT", "SOL/USDT"]
DATA_DIR = Path.cwd().parent / "data"
exchange = ccxt.binance()

In [ ]:
summary = []

for element in WATCHLIST :
    candles=fetch_ohlcv_history(exchange,
                        element)
    
    df=candles_to_dataframe(candles)

    report = validate_ohlcv(df)

    df = drop_incomplete_candles(df, exchange.milliseconds())

    raw_path, processed_path=save_dataset(candles, df, element, DATA_DIR)

    reloaded=load_processed(element, DATA_DIR)

    pd.testing.assert_frame_equal(df, reloaded)

    problems = {
        check: result
        for check, result in report.items()
        if (check == "sorted" and result is not True)
        or (check != "sorted" and result != 0)
    }

    summary.append({
        "symbol": element,
        "raw_rows": len(candles),
        "processed_rows": len(df),
        "first_date": df.index[0].date(),
        "last_date": df.index[-1].date(),
        "problems": problems,
    })

pd.DataFrame(summary)

,symbol,raw_rows,processed_rows,first_date,last_date,problems
0,BTC/USDT,3349,3348,2017-08-17,2026-10-16,{}
1,ETH/USDT,3349,3348,2017-08-17,2026-10-16,{}
2,SOL/USDT,2259,2258,2020-08-11,2026-10-16,{}
